# With QuTiP and other code, generate an advantageous quantum distribution, check that it is no-signaling, and compute the CHSH values for both short and long paths.

## Imports

In [ ]:
import numpy as np
import qutip as qt

## Variables

### States

In [ ]:
ket_0 = qt.basis(2, 0)  # Create a basis state |0>
ket_1 = qt.basis(2, 1)  # Create a basis state |1>

# ket_plus = (ket_0 + ket_1).unit()  # Create the superposition state |+>
# ket_minus = (ket_0 - ket_1).unit()  # Create the superposition state |- >

ket_00 = qt.tensor(ket_0, ket_0)  # Create the state |00>
ket_11 = qt.tensor(ket_1, ket_1)  # Create the state |11>
ket_phi = (ket_00 + ket_11).unit()  # Create the Bell state |Φ+>

### Operators

In [ ]:
sigma_x = qt.sigmax()  # Pauli X operator
sigma_z = qt.sigmaz()  # Pauli Z operator

sigma_x_plus_z = (sigma_x + sigma_z).unit()  # Create the operator (σx + σz)/√2
sigma_x_minus_z = (sigma_x - sigma_z).unit()  # Create the operator (σx - σz)/√2

## Set up routed setting

### Distribution values

In [ ]:
z_short = np.zeros((2,2,2,2), dtype=complex) # Indexed a,b,x,y, short-path correlations
z_long = np.zeros((2,2,2,2), dtype=complex) # Same, but long-path correlations

### BB84 typed protocol

In [ ]:
alice_measurements = [sigma_x, sigma_z]
bob_short_measurements = [sigma_x_plus_z, sigma_x_minus_z]
bob_long_measurements = [sigma_z, sigma_x]

alice_eigenstates = [
    [alice_measurements[0].eigenstates()[1][0], alice_measurements[0].eigenstates()[1][1]],
    [alice_measurements[1].eigenstates()[1][0], alice_measurements[1].eigenstates()[1][1]],
]
bob_short_eigenstates = [
    [bob_short_measurements[0].eigenstates()[1][0], bob_short_measurements[0].eigenstates()[1][1]],
    [bob_short_measurements[1].eigenstates()[1][0], bob_short_measurements[1].eigenstates()[1][1]],
]
bob_long_eigenstates = [
    [bob_long_measurements[0].eigenstates()[1][0], bob_long_measurements[0].eigenstates()[1][1]],
    [bob_long_measurements[1].eigenstates()[1][0], bob_long_measurements[1].eigenstates()[1][1]],
]

## Compute the distribution

### Distribution coordinates

In [ ]:
for a,b,x,y in [
    (_a,_b,_x,_y)
    for _a in range(2) for _b in range(2) for _x in range(2) for _y in range(2)
    ]:
    # Short-path correlations
    P_A = alice_eigenstates[x][a] @ alice_eigenstates[x][a].dag()
    P_B = bob_short_eigenstates[y][b] @ bob_short_eigenstates[y][b].dag()
    P_joint = qt.tensor(P_A, P_B)
    z_short[a,b,x,y] = ket_phi.dag() @ P_joint @ ket_phi # Not type hinted properly in __matmul__ definition for Qobj,
    # but a matmul outputting a scalar casts answer to a scalar type.

    # Long-path correlations
    P_A = alice_eigenstates[x][a] @ alice_eigenstates[x][a].dag()
    P_B = bob_long_eigenstates[y][b] @ bob_long_eigenstates[y][b].dag()
    P_joint = qt.tensor(P_A, P_B)
    z_long[a,b,x,y] = ket_phi.dag() @ P_joint @ ket_phi # Same as above. The result is being cast to scalar.


### Cast to real matrix, checking that the values are real

In [ ]:
# Check that the values are real
assert np.all(np.isreal(z_short)), "Short-path correlations contain complex values"
assert np.all(np.isreal(z_long)), "Long-path correlations contain complex values"


z_short = z_short.real  # Reshape and cast to real matrix
z_long = z_long.real


In [ ]:
print("Short-path correlations:")
print(z_short.reshape(4, 4))
print("\nLong-path correlations:")
print(z_long.reshape(4, 4))

# Check soundness of the results

In [ ]:
from behaviors import RoutedBehavior

In [ ]:
flattened_distribution = np.concatenate(
    [z_short.flatten(), z_long.flatten()]
)  # Flatten the distribution for RoutedBehavior

experiment = RoutedBehavior(delta=2, m=2, vector=flattened_distribution)

### Make checks that the distribution is normalized, no-signaling

In [ ]:
print("Experiment distribution:", experiment)
print("Experiment is valid NS:", experiment.is_no_signaling())

# Compute the CHSH value

## Bell correlators computation as a function

In [ ]:
def correlator(idx_A: int, idx_B: int, matrix: np.ndarray) -> float:
    """
    Compute the Bell correlator <A_{idx_A} B_{idx_B}> for the given matrix.
    Matrix should have 16=delta^2 * m^2 elements, indexed as (a,b,x,y).
    Expected shape is (2, 2, 2, 2).

    :param idx_A: Index for Alice's measurement (0 or 1).
    :param idx_B: Index for Bob's measurement (0 or 1).
    :param matrix: The correlation matrix with shape (2, 2, 2, 2).
    :return: The computed Bell correlator value.
    """
    corr_value = 0
    for a,b in [(a,b) for a in range(2) for b in range(2)]:
        corr_value += (-1)**(a+b) * matrix[a,b,idx_A,idx_B]
    return corr_value


def chsh_value(matrix: np.ndarray) -> float:
    """
    Compute the CHSH value for the given correlation matrix.
    Matrix should have 16=delta^2 * m^2 elements, indexed as (a,b,x,y).
    Expected shape is (2, 2, 2, 2).

    :param matrix: The correlation matrix with shape (2, 2, 2, 2).
    :return: The computed CHSH value.
    """
    return (
        correlator(0, 0, matrix) + correlator(0, 1, matrix) +
        correlator(1, 0, matrix) - correlator(1, 1, matrix)
    )

## Apply the function to short and long-range distributions

In [ ]:
print(f"Short-path CHSH value: {chsh_value(z_short)}")
print(f"Long-path CHSH value: {chsh_value(z_long)}")

# Decompose over boxworld probability distributions

## Import modules for linear programming

In [ ]:
from loguru import logger
from scipy.optimize import OptimizeResult, linprog


## Import boxworld vertices

In [ ]:
vertices_file = "../pruned_vertices_222.txt"  # File containing the vertices

vertices_list: list[np.ndarray] = []
seen_vertices = set()

logger.info(f"Loading vertices from {vertices_file}...")
with open(vertices_file, "r") as f:
    for line in f:
        if line.strip() and line.strip() not in seen_vertices:
            vertex = np.fromstring(line.strip(), sep=",", dtype=float)
            vertices_list.append(vertex)
            seen_vertices.add(line.strip())
logger.info(f"Loaded {len(vertices_list)} vertices.")

vertices_arr = np.array(vertices_list)

## Set up a function to compute the convex combination of boxworld vertices giving a given distribution

In [ ]:
def convex_combination(
    distribution: np.ndarray,
    vertices: np.ndarray
) -> np.ndarray:
    """
    Compute the convex combination of vertices that approximates the given distribution.

    :param distribution: The target distribution to approximate.
    :param vertices: The vertices to use for the convex combination.
    :return: The coefficients of the convex combination.
    """
    A = np.vstack((np.ones((1, vertices.shape[0])), vertices.T))
    b = np.hstack((1, distribution))

    res:OptimizeResult = linprog(
        c=np.zeros(vertices.shape[0]),
        A_eq=A,
        b_eq=b,
        bounds=(0, 1),
        method='highs'
    )

    if res.success:
        logger.info("Convex combination found successfully.")
        return res.x
    else:
        logger.error("Failed to find a convex combination.")
        raise ValueError("Convex combination optimization failed: " + res.message)


## Compute the convex combination of boxworld vertices

In [ ]:
advantageous_q_distribution = experiment.get_vector()
assert advantageous_q_distribution is not None, "The distribution vector is None."

coefficients = convex_combination(advantageous_q_distribution, vertices_arr)

### Get non-null coefficients and matching vertices


In [ ]:
non_null_indices = np.where(coefficients > 0)[0]
matching_vertices = vertices_arr[non_null_indices]

In [ ]:
print("Non-null indices:", non_null_indices)
print("Non-null coefficients:", coefficients[non_null_indices])
# print("Matching vertices:", matching_vertices)

In [ ]:
def quickload_file_with_info(exp_file: str) -> tuple[list[str],list[np.ndarray]]:
    seen = set()  # To deduplicate points
    vertices_list = []  # To store the vertices
    info_list = []
    # Load the vertices from the file
    logger.info(f"Loading vertices from {exp_file}...")

    with open(exp_file, "r") as f:
        while True:
            line = f.readline()
            if not line:
                break
            # Convert the line to a numpy array and deduplicate
            info, point_str = line.split(";")
            point = np.array([float(x) for x in point_str.strip().split(",")])

            if point.tobytes() not in seen:
                seen.add(point.tobytes())
                info_list.append(info)
                vertices_list.append(point)

    logger.info(f"Loaded {len(vertices_list)} unique vertices from {exp_file}")

    return info_list, vertices_list


In [ ]:
info_list, ann_vertices = quickload_file_with_info("../222_with_info_not_pruned.txt")

In [ ]:
ann_vertices_tupled = [tuple(vertex) for vertex in ann_vertices]
combination_str = ""
combination_arr = np.zeros(matching_vertices.shape[1])

for i, hull_vertex in enumerate(list(matching_vertices)):
    vertex_index_in_annotated = ann_vertices_tupled.index(tuple(hull_vertex))
    combination_str += f"- {coefficients[non_null_indices[i]]}\n   - {info_list[vertex_index_in_annotated]}\n   - {hull_vertex.tolist()}\n"  # noqa: E501
    combination_arr += coefficients[non_null_indices[i]] * hull_vertex


In [ ]:
print("Combination array:", RoutedBehavior(delta=2, m=2, vector=combination_arr))
assert advantageous_q_distribution is not None, "Original array is None"
print("Original array:", RoutedBehavior(delta=2, m=2, vector=advantageous_q_distribution))
print()
print("ARE EQUAL:", np.allclose(combination_arr, advantageous_q_distribution, atol=1e-6))

In [ ]:
print("#### Combinations:")
print(combination_str)

#### Combinations:
- 0.07322330470336309
   - State: w1, Alice: ['zero', 'zero'], Bob: ['zero', 'zero'], Long Bob: u, Transform: f13
   - [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0]
- 0.07322330470336291
   - State: w1, Alice: ['zero', 'zero'], Bob: ['zero', 'u'], Long Bob: u, Transform: f13
   - [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0]
- 0.07322330470336315
   - State: w1, Alice: ['u', 'u'], Bob: ['u', 'zero'], Long Bob: u, Transform: f1
   - [1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
- 0.07322330470336297
   - State: w1, Alice: ['u', 'u'], Bob: ['u', 'u'], Long Bob: u, Transform: f1
   - [1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
- 0.07322330470336302
   - State: w1, Alice: ['zero', 'u'], Bob: ['u', 'zero'], Long Bob: u, Transform: f5
   - [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
- 0.07322330470336313
   - State: w1, Alice: ['u', 'zero'], Bob: ['zero', 'u'], Long Bob: u, Transform: f9
   - [0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
- 0.14644660940672616
   - State: w17, Alice: ['e1', 'e2'], Bob: ['e2', 'e2'], Long Bob: e1, Transform: f13
   - [0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.5, 0.5, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5]
- 0.3535533905932735
   - State: w17, Alice: ['e1', 'e2'], Bob: ['e3', 'e2'], Long Bob: e2, Transform: f10
   - [0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.0, 0.5, 0.5, 0.0, 0.5, 0.0, 0.0, 0.5, 0.5, 0.0, 0.0, 0.5, 0.0, 0.5, 0.5, 0.0]
- 0.06066017177982186
   - State: w17, Alice: ['e1', 'e2'], Bob: ['e3', 'e2'], Long Bob: e1, Transform: f13
   - [0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5]

In [ ]:
idx = [0,1,2,3,4,5,7,8]
arbitrary_combination = np.zeros(32)
for i in idx:
    arbitrary_combination += coefficients[non_null_indices][i]/np.sum(coefficients[non_null_indices][idx]) * matching_vertices[i,:]
arbitrary_behavior = RoutedBehavior(delta=2, m=2, vector=arbitrary_combination)
print(arbitrary_behavior)  # Sum of the matching vertices
print(coefficients[non_null_indices][idx])  # Coefficients of the convex combination
print("C_short", chsh_value(arbitrary_combination[:16].reshape(2, 2, 2, 2)))
print("C_long", chsh_value(arbitrary_combination[16:].reshape(2, 2, 2, 2)))

In [ ]:
input_vec = np.array([0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5])
sanitycheck_behavior_distrib = RoutedBehavior(delta=2, m=2, vector=input_vec)

print(sanitycheck_behavior_distrib)

In [ ]:
p1 = 0.14644660940672616
a1 = [0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.5, 0.5, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5]
p2 = 0.3535533905932735
a2 = [0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.0, 0.5, 0.5, 0.0, 0.5, 0.0, 0.0, 0.5, 0.5, 0.0, 0.0, 0.5, 0.0, 0.5, 0.5, 0.0]
p3 = 0.06066017177982186
a3 = [0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.5, 0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5]

a_tot = (p1 * np.array(a1) + p2 * np.array(a2) + p3 * np.array(a3)) / (p1 + p2 + p3)
print(RoutedBehavior(delta=2, m=2, vector=a_tot))

print("CHSH short :", chsh_value(a_tot[:16].reshape(2, 2, 2, 2)))
print("CHSH long :", chsh_value(a_tot[16:].reshape(2, 2, 2, 2)))